In [25]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import json
import pandas as pd

In [26]:
df_business_raw = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_business.json",
    pandas_kwargs={"lines": True},
)

# only care about Santa Barbara for now
df_business = df_business_raw[df_business_raw["city"]== "Santa Barbara"].copy()


In [27]:
# the categories that we want to include

allowed_categories = set(
    pd.read_csv("../processed/categories.csv")["category"]
    .dropna()
    .str.strip()
)

len(allowed_categories)

24

In [28]:
category_boolean = df_business["categories"].fillna("").apply(
    lambda value: any(
        category.strip() in allowed_categories
        for category in value.split(",")
    )
)

category_boolean

0         False
26        False
85         True
91        False
120       False
          ...  
150239    False
150264     True
150284    False
150290    False
150303    False
Name: categories, Length: 3829, dtype: bool

In [29]:
df_business = df_business.loc[category_boolean].copy()

In [30]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,1,"{'RestaurantsTakeOut': 'True', 'NoiseLevel': '...","Food, Restaurants, Salad, Coffee & Tea, Breakf...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-14:0', '..."
141,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,34.408715,-119.685019,4.0,2404,1,"{'OutdoorSeating': 'True', 'RestaurantsAttire'...","Live/Raw Food, Restaurants, Seafood, Beer Bar,...","{'Monday': '0:0-0:0', 'Tuesday': '11:0-21:0', ..."
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,1,"{'RestaurantsPriceRange2': '2', 'BusinessAccep...","Coffee & Tea, Coffee Roasteries, Food","{'Monday': '6:0-17:0', 'Tuesday': '6:0-17:0', ..."
946,-ujBP1Dw0j1-Ffaz97-LXQ,Lama Dog Tap Room,116 Santa Barbara St,Santa Barbara,CA,93101,34.415747,-119.688480,4.0,238,1,"{'RestaurantsReservations': 'False', 'WiFi': '...","Beer, Wine & Spirits, Pubs, Arts & Entertainme...","{'Monday': '0:0-0:0', 'Tuesday': '11:30-17:0',..."
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,1,"{'WheelchairAccessible': 'True', 'WiFi': 'u'fr...","Themed Cafes, Cafes, Pets, Arts & Entertainmen...","{'Monday': '10:0-17:15', 'Wednesday': '10:0-19..."


In [31]:
from pathlib import Path
from pprint import pformat

business = df_business.loc[
    df_business["name"].str.strip().eq("Cat Therapy")
].iloc[0]

output_path = Path("business_example_Cat_Therapy.txt")
output_path.write_text(
    pformat(business["attributes"], sort_dicts=False),
    encoding="utf-8",
)

431

In [32]:
attribute_keys = (
    pd.read_csv("attributes.csv")["attribute"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Handle missing dictionaries without changing the original column.
safe_attributes = df_business["attributes"].apply(
    lambda value: value if isinstance(value, dict) else {}
)

attributes_df = pd.DataFrame(
    safe_attributes.tolist(),
    index=df_business.index,
).reindex(columns=attribute_keys)

for key in attribute_keys:
    df_business[f"attribute_{key}"] = attributes_df[key]

In [33]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attribute_RestaurantsCounterService,attribute_RestaurantsDelivery,attribute_RestaurantsGoodForGroups,attribute_RestaurantsPriceRange2,attribute_RestaurantsReservations,attribute_RestaurantsTableService,attribute_RestaurantsTakeOut,attribute_Smoking,attribute_WheelchairAccessible,attribute_WiFi
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,...,NaN,None,True,2,False,False,True,NaN,True,u'no'
141,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,34.408715,-119.685019,4.0,2404,...,NaN,True,False,2,False,NaN,True,NaN,NaN,u'no'
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,...,NaN,NaN,NaN,2,NaN,NaN,True,NaN,NaN,u'free'
946,-ujBP1Dw0j1-Ffaz97-LXQ,Lama Dog Tap Room,116 Santa Barbara St,Santa Barbara,CA,93101,34.415747,-119.688480,4.0,238,...,NaN,True,True,2,False,NaN,True,u'no',True,'free'
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,...,NaN,NaN,NaN,1,NaN,NaN,False,NaN,True,u'free'


In [34]:
category_names = (
    pd.read_csv("../processed/categories.csv")["category"])

allowed_categories = set(category_names)

# Split each business's categories and remove surrounding spaces.
business_categories = df_business["categories"].fillna("").apply(
    lambda value: {
        category.strip()
        for category in value.split(",")
        if category.strip()
    }
)

# True when the business has that exact category.
for category in category_names:
    df_business[f"category_{category}"] = business_categories.apply(
        lambda values: category in values
    )

# True when at least one category is outside the CSV list.
df_business["category_other"] = business_categories.apply(
    lambda values: bool(values - allowed_categories)
)

In [35]:
categories_exclude = pd.read_csv("../processed/categories_exclude.csv")
excluded = set(categories_exclude["category"])

df_business = df_business[
    ~df_business["categories"].fillna("").apply(
        lambda value: any(
            category.strip() in excluded
            for category in value.split(",")
        )
    )
]

In [36]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,category_Beer Hall,category_Cideries,category_Bookstores,category_Used Bookstore,category_Community Centers,category_Lounges,category_Wine Bars,category_Pubs,category_Gastropubs,category_other
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,...,False,False,False,False,False,False,False,False,False,True
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,...,False,False,False,False,False,False,False,False,False,True
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,...,False,False,False,False,False,False,False,False,False,True
1835,aYMpjij5ShtEoZueMrQPRw,The Mill,406 E Haley St,Santa Barbara,CA,93101,34.421119,-119.690633,4.0,5,...,False,False,False,False,False,False,False,False,False,True
1848,PaqDZUu76fZtMvcYqCEBig,Rincon Brewery,205 Santa Barbara St,Santa Barbara,CA,93101,34.416182,-119.689776,4.0,32,...,False,False,False,False,False,False,False,False,False,True


In [37]:
days = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]

# Handle missing hours without changing the original column.
safe_hours = df_business["hours"].apply(
    lambda value: value if isinstance(value, dict) else {}
)

for day in days:
    day_hours = safe_hours.apply(lambda hours: hours.get(day))

    split_hours = (
        day_hours.astype("string")
        .str.split("-", n=1, expand=True)
        .reindex(columns=[0, 1])
    )

    df_business[f"{day}_start"] = split_hours[0]
    df_business[f"{day}_end"] = split_hours[1]

In [38]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,Wednesday_start,Wednesday_end,Thursday_start,Thursday_end,Friday_start,Friday_end,Saturday_start,Saturday_end,Sunday_start,Sunday_end
85,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,...,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0
691,-kY_HDP7IMvGl-kBIZVU4A,Dune Coffee Roasters - Anacapa,528 Anacapa St,Santa Barbara,CA,93101,34.418994,-119.695068,4.0,320,...,6:0,17:0,6:0,17:0,6:0,17:0,6:0,17:0,7:0,17:0
1320,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,...,10:0,19:0,10:0,19:0,10:0,20:0,10:0,20:0,10:0,19:0
1835,aYMpjij5ShtEoZueMrQPRw,The Mill,406 E Haley St,Santa Barbara,CA,93101,34.421119,-119.690633,4.0,5,...,11:0,22:0,11:0,22:0,11:0,22:30,11:0,22:30,11:0,22:0
1848,PaqDZUu76fZtMvcYqCEBig,Rincon Brewery,205 Santa Barbara St,Santa Barbara,CA,93101,34.416182,-119.689776,4.0,32,...,<NA>,<NA>,11:30,21:0,11:30,21:0,11:30,21:0,11:30,19:0


In [39]:
#df_reviews = kagglehub.dataset_load(
#    KaggleDatasetAdapter.PANDAS,
#    "yelp-dataset/yelp-dataset",
#    "yelp_academic_dataset_review.json")


In [40]:
business_ids = set(df_business["business_id"])

# Download the source file or reuse its cached copy.
reviews_path = kagglehub.dataset_download(
    "yelp-dataset/yelp-dataset",
    path="yelp_academic_dataset_review.json",
)

matching_chunks = []

# Read 10,000 reviews at a time.
with pd.read_json(
    reviews_path,
    lines=True,
    chunksize=10_000,
) as reader:
    for chunk in reader:
        selected = chunk.loc[
            chunk["business_id"].isin(business_ids)
        ]

        if not selected.empty:
            matching_chunks.append(selected.copy())

# Combine only the matching reviews.
df_reviews = (
    pd.concat(matching_chunks, ignore_index=True)
    if matching_chunks
    else pd.DataFrame()
)

del matching_chunks

print(f"Loaded {len(df_reviews):,} matching reviews.")
df_reviews.head(10)

Loaded 27,128 matching reviews.


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,L0jv8c2FbpWSlfNC6bbUEA,bFPdtzu11Oi0f92EAcjqmg,IDtLPgUrqorrpqSLdfMhZQ,5,0,0,0,What a great addition to the Funk Zone! Grab ...,2016-10-13 22:50:47
1,SpmKsFnwg35PTgCVujOOJA,RiQWbgwR-asiU_mA7s8gpg,pUVRCYL8rT4I5Ry5FYkRsA,5,2,0,1,Sweet spot in the Waterline. Came for some liv...,2016-12-19 22:58:19
2,Gczff1Qxg4WVmlBLPrZwzQ,cpp89UW0imv45y3a_O444w,34Eqv8jXgxg_EEwcsNgeeg,5,0,0,0,We absolutely love everything we have tried he...,2014-05-25 22:33:02
3,bi6GaeWDGceGv62lXTIKQA,RgtbLaiU22zqaCk20HgbiQ,bjhCtlYHrkgA5Ku8l-rB3g,1,1,0,0,Very disappointed. We went to eat at 2:15 on ...,2015-01-04 02:26:46
4,H4mSxGeKTv4VFcc9YUmtpQ,oGV84fsuRbLQRRQ6ZZC1dA,-kY_HDP7IMvGl-kBIZVU4A,4,0,0,0,The espresso here is excellent. My favorite pl...,2016-07-14 19:43:42
5,LFpaQzYkP5Pzm5lEjJpTRw,Ttb21DpWqphQMNSS5TXgMA,IDtLPgUrqorrpqSLdfMhZQ,5,0,0,0,Possibly the best breakfast sandwich EVER. On...,2017-01-15 00:36:04
6,s_X5uNLjLSgK_itDNFiadg,NZdASLiCRqQI1yIl9MIY5w,MbzgGsMQpGyVrUJXi_Jw0Q,4,0,0,0,I am from Colorado and am visiting Santa Barba...,2018-03-15 19:16:02
7,Q3fPo_x6xKxafAzy1hFITg,ha2Lv7WnxvdYnRsqwL1apQ,IDtLPgUrqorrpqSLdfMhZQ,4,0,0,0,Pricey ( a ham and cheese croissant was $5.50)...,2018-07-17 00:20:41
8,NoPOfj7gC1l48dC0XH7Spw,tgP2gKsf3O9MyuLuolv2oQ,KiE0h68HGOO7ZXAqkMBdiw,1,0,1,0,"I went in when they had 4 people working, wait...",2016-01-30 01:10:42
9,u6EW2edED7ZfbZAw7lcYYQ,dNoGiISFXS1YM5HacSh0eA,bjhCtlYHrkgA5Ku8l-rB3g,5,0,0,0,Great breads especially the German rye. My fa...,2013-11-14 02:00:49


In [41]:
df_reviews.shape 

(27128, 9)

In [42]:
df_union = df_reviews.merge(df_business, on="business_id", how="left")

In [43]:
df_union.head()

,review_id,user_id,business_id,stars_x,useful,funny,cool,text,date,name,...,Wednesday_start,Wednesday_end,Thursday_start,Thursday_end,Friday_start,Friday_end,Saturday_start,Saturday_end,Sunday_start,Sunday_end
0,L0jv8c2FbpWSlfNC6bbUEA,bFPdtzu11Oi0f92EAcjqmg,IDtLPgUrqorrpqSLdfMhZQ,5,0,0,0,What a great addition to the Funk Zone! Grab ...,2016-10-13 22:50:47,Helena Avenue Bakery,...,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0
1,SpmKsFnwg35PTgCVujOOJA,RiQWbgwR-asiU_mA7s8gpg,pUVRCYL8rT4I5Ry5FYkRsA,5,2,0,1,Sweet spot in the Waterline. Came for some liv...,2016-12-19 22:58:19,Topa Topa Brewing Company,...,12:0,22:0,12:0,22:0,12:0,23:0,12:0,23:0,12:0,20:0
2,Gczff1Qxg4WVmlBLPrZwzQ,cpp89UW0imv45y3a_O444w,34Eqv8jXgxg_EEwcsNgeeg,5,0,0,0,We absolutely love everything we have tried he...,2014-05-25 22:33:02,Backyard Bowls,...,8:0,17:0,8:0,17:0,8:0,17:0,8:0,17:0,8:0,17:0
3,bi6GaeWDGceGv62lXTIKQA,RgtbLaiU22zqaCk20HgbiQ,bjhCtlYHrkgA5Ku8l-rB3g,1,1,0,0,Very disappointed. We went to eat at 2:15 on ...,2015-01-04 02:26:46,Our Daily Bread,...,6:30,15:30,6:30,15:30,6:30,15:30,6:30,15:30,6:30,16:0
4,H4mSxGeKTv4VFcc9YUmtpQ,oGV84fsuRbLQRRQ6ZZC1dA,-kY_HDP7IMvGl-kBIZVU4A,4,0,0,0,The espresso here is excellent. My favorite pl...,2016-07-14 19:43:42,Dune Coffee Roasters - Anacapa,...,6:0,17:0,6:0,17:0,6:0,17:0,6:0,17:0,7:0,17:0


In [44]:
df_union.columns.tolist()

['review_id',
 'user_id',
 'business_id',
 'stars_x',
 'useful',
 'funny',
 'cool',
 'text',
 'date',
 'name',
 'address',
 'city',
 'state',
 'postal_code',
 'latitude',
 'longitude',
 'stars_y',
 'review_count',
 'is_open',
 'attributes',
 'categories',
 'hours',
 'attribute_AcceptsInsurance',
 'attribute_AgesAllowed',
 'attribute_Alcohol',
 'attribute_Ambience',
 'attribute_BYOB',
 'attribute_BYOBCorkage',
 'attribute_BestNights',
 'attribute_BikeParking',
 'attribute_BusinessAcceptsBitcoin',
 'attribute_BusinessAcceptsCreditCards',
 'attribute_BusinessParking',
 'attribute_ByAppointmentOnly',
 'attribute_Caters',
 'attribute_CoatCheck',
 'attribute_Corkage',
 'attribute_DietaryRestrictions',
 'attribute_DogsAllowed',
 'attribute_DriveThru',
 'attribute_GoodForDancing',
 'attribute_GoodForKids',
 'attribute_GoodForMeal',
 'attribute_HairSpecializesIn',
 'attribute_HappyHour',
 'attribute_HasTV',
 'attribute_Music',
 'attribute_NoiseLevel',
 'attribute_Open24Hours',
 'attribute_Outdo

In [45]:
from pathlib import Path

print(Path.cwd())

/Users/rejaimes/repos/coffee-project/data/raw


In [46]:
output_path = Path("../processed/santa_barbara_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_union.to_csv(output_path, index=False)

print(f"Saved {len(df_union):,} rows to {output_path.resolve()}")

Saved 27,128 rows to /Users/rejaimes/repos/coffee-project/data/processed/santa_barbara_data.csv
